In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## Hospital Admission results

In [6]:
def process_hosp_admission_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in([ST.ADMISSION]),
        expected_expr=pl.col("expected").is_in([ST.ADMISSION]),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        **kwargs,
    )

In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.ed import HospitalAdmissionAtTriageDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = HospitalAdmissionAtTriageDataset(path)
sum_adm = sum([d[1]['expected'] for d in dataset])
sum_adm, len(dataset), sum_adm / len(dataset)

63944 63944


(30620, 63944, 0.47885649943700737)

In [3]:
import json
true_ids = [d[1]['hadm_id'] for d in dataset if d[1]['expected'] == True]
with open("true_ids.json", "w") as f:
    json.dump(true_ids, f)


In [ ]:
hosp_admission_results = []
for input_dir in (results_dir / Task.ED_HOSPITALIZATION).iterdir():
    df = process_hosp_admission_results(input_dir)
    rep_num = df["counts"].mean()
    res = compute_and_print_metrics(
        df["expected"],
        df["actual"],
        f"Hospital Admission (rep_num={rep_num:.2f})\n{input_dir.name}",
    )
    hosp_admission_results.append(
        {
            "name": input_dir.name,
            "auc": res["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.show()
hosp_admission_results = pl.DataFrame(hosp_admission_results).sort("auc", descending=True)

In [ ]:
hosp_admission_results